# Airline Passenger Satisfaction Analysis

This notebook analyzes the airline passenger satisfaction dataset using SQL. The raw CSV is loaded into a local SQLite database, and all 25 analytical questions below are answered with real SQL queries executed against that database.

Visualizations are not included here. They live in a separate Plotly Dash application (`app.py`) that ships alongside this notebook as a standalone interactive dashboard.

**Structure**

1. Setup and data loading
2. Data overview
3. Demographics and customer profiles
4. Overall satisfaction and loyalty
5. Service ratings and performance
6. Delays and operational impact
7. Key takeaways

## 1. Setup

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


def run_sql(query, connection):
    return pd.read_sql_query(query, connection)


def show_result(df, caption):
    numeric_cols = df.select_dtypes(include='number').columns
    styled = df.style.set_caption(caption)
    if len(numeric_cols) > 0:
        styled = styled.background_gradient(cmap='Blues', subset=numeric_cols)
    styled = styled.set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '15px'), ('font-weight', '600'), ('text-align', 'left'), ('padding', '6px 2px'), ('color', '#1a1a2e')]},
        {'selector': 'th', 'props': [('background-color', '#1a1a2e'), ('color', 'white'), ('font-weight', '600'), ('padding', '8px 14px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('padding', '6px 14px')]},
    ])
    return styled.hide(axis='index')

## 2. Loading the Data into SQLite

In [2]:
csv_path = Path('train.csv')
db_path = Path('airline_satisfaction.db')

df = pd.read_csv(csv_path)
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')])

conn = sqlite3.connect(db_path)
df.to_sql('flights', conn, if_exists='replace', index=False)

conn.execute('DROP VIEW IF EXISTS flights_flagged')
conn.execute('''
CREATE VIEW flights_flagged AS
SELECT *, CASE WHEN satisfaction = 'satisfied' THEN 1 ELSE 0 END AS satisfaction_flag
FROM flights
''')
conn.commit()

service_cols = [
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Inflight service',
    'Cleanliness',
]

print(f'Loaded {len(df):,} rows into the flights table of {db_path.name}')

Loaded 103,904 rows into the flights table of airline_satisfaction.db


## 3. Data Overview

In [3]:
overview_query = '''
SELECT
    COUNT(*) AS total_passengers,
    COUNT(DISTINCT Class) AS travel_classes,
    ROUND(AVG(Age), 1) AS avg_age,
    ROUND(AVG("Flight Distance"), 1) AS avg_flight_distance
FROM flights;
'''
overview = run_sql(overview_query, conn)
show_result(overview, 'Dataset overview')

total_passengers,travel_classes,avg_age,avg_flight_distance
103904,3,39.400000,1189.400000


In [4]:
sample_query = '''
SELECT *
FROM flights
LIMIT 5;
'''
sample = run_sql(sample_query, conn)
sample

,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,satisfied
3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,2,2,2,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,4,5,5,3,3,4,4,3,3,3,0,0.0,satisfied


## 4. Demographics and Customer Profiles

This section profiles who is flying: age distribution, travel class mix, loyalty patterns, and how distance and age vary across customer segments.

### Q1. What is the passenger distribution across different age groups, and how does the proportion of loyal customers vary by age bracket?

In [5]:
query_1 = '''
SELECT
    CASE
        WHEN Age < 18 THEN '0-17'
        WHEN Age < 30 THEN '18-29'
        WHEN Age < 45 THEN '30-44'
        WHEN Age < 60 THEN '45-59'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS passenger_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM flights), 2) AS pct_of_total,
    ROUND(100.0 * SUM(CASE WHEN "Customer Type" = 'Loyal Customer' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_loyal
FROM flights
GROUP BY age_group
ORDER BY MIN(Age);
'''
result_1 = run_sql(query_1, conn)
show_result(result_1, 'Age group distribution and loyalty share')

age_group,passenger_count,pct_of_total,pct_loyal
0-17,7931,7.630000,85.850000
18-29,22796,21.940000,55.690000
30-44,32943,31.710000,82.460000
45-59,30515,29.370000,94.890000
60+,9719,9.350000,95.650000


In [6]:
top = result_1.loc[result_1['pct_loyal'].idxmax()]
low = result_1.loc[result_1['pct_loyal'].idxmin()]
print(f"{top['age_group']} has the highest loyalty share at {top['pct_loyal']}%, while {low['age_group']} has the lowest at {low['pct_loyal']}%")

60+ has the highest loyalty share at 95.65%, while 18-29 has the lowest at 55.69%


### Q2. What percentage of the total passenger base belongs to each travel class, and how does this breakdown differ by gender?

In [7]:
query_2 = '''
SELECT
    Class,
    Gender,
    COUNT(*) AS passenger_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM flights), 2) AS pct_of_total
FROM flights
GROUP BY Class, Gender
ORDER BY Class, Gender;
'''
result_2 = run_sql(query_2, conn)
show_result(result_2, 'Travel class share by gender')

Class,Gender,passenger_count,pct_of_total
Business,Female,24927,23.990000
Business,Male,24738,23.810000
Eco,Female,23858,22.960000
Eco,Male,22887,22.030000
Eco Plus,Female,3942,3.790000
Eco Plus,Male,3552,3.420000


In [8]:
top = result_2.loc[result_2['pct_of_total'].idxmax()]
print(f"The largest single segment is {top['Gender']} passengers in {top['Class']} class at {top['pct_of_total']}% of all passengers")

The largest single segment is Female passengers in Business class at 23.99% of all passengers


### Q3. What is the ratio of loyal customers to disloyal customers when comparing Business travel versus Personal travel?

In [9]:
query_3 = '''
SELECT
    "Type of Travel",
    SUM(CASE WHEN "Customer Type" = 'Loyal Customer' THEN 1 ELSE 0 END) AS loyal_count,
    SUM(CASE WHEN "Customer Type" != 'Loyal Customer' THEN 1 ELSE 0 END) AS disloyal_count,
    ROUND(1.0 * SUM(CASE WHEN "Customer Type" = 'Loyal Customer' THEN 1 ELSE 0 END) /
          NULLIF(SUM(CASE WHEN "Customer Type" != 'Loyal Customer' THEN 1 ELSE 0 END), 0), 2) AS loyal_to_disloyal_ratio
FROM flights
GROUP BY "Type of Travel";
'''
result_3 = run_sql(query_3, conn)
show_result(result_3, 'Loyal to disloyal ratio by travel purpose')

Type of Travel,loyal_count,disloyal_count,loyal_to_disloyal_ratio
Business travel,52838,18817,2.810000
Personal Travel,32085,164,195.640000


In [10]:
for _, row in result_3.iterrows():
    print(f"{row['Type of Travel']}: {row['loyal_to_disloyal_ratio']} loyal customers for every disloyal customer")

Business travel: 2.81 loyal customers for every disloyal customer
Personal Travel: 195.64 loyal customers for every disloyal customer


### Q4. How does the average flight distance compare between loyal customers and disloyal customers?

In [11]:
query_4 = '''
SELECT
    "Customer Type",
    ROUND(AVG("Flight Distance"), 1) AS avg_flight_distance,
    COUNT(*) AS passenger_count
FROM flights
GROUP BY "Customer Type";
'''
result_4 = run_sql(query_4, conn)
show_result(result_4, 'Average flight distance by customer type')

Customer Type,avg_flight_distance,passenger_count
Loyal Customer,1295.600000,84923
disloyal Customer,714.500000,18981


In [12]:
loyal = result_4[result_4['Customer Type'] == 'Loyal Customer']['avg_flight_distance'].iloc[0]
disloyal = result_4[result_4['Customer Type'] == 'disloyal Customer']['avg_flight_distance'].iloc[0]
print(f"Loyal customers fly an average of {loyal} miles versus {disloyal} miles for disloyal customers, a gap of {round(loyal - disloyal, 1)} miles")

Loyal customers fly an average of 1295.6 miles versus 714.5 miles for disloyal customers, a gap of 581.1 miles


### Q5. What is the average age of passengers across the three different travel classes?

In [13]:
query_5 = '''
SELECT
    Class,
    ROUND(AVG(Age), 1) AS avg_age,
    COUNT(*) AS passenger_count
FROM flights
GROUP BY Class
ORDER BY avg_age DESC;
'''
result_5 = run_sql(query_5, conn)
show_result(result_5, 'Average age by travel class')

Class,avg_age,passenger_count
Business,41.600000,49665
Eco Plus,38.700000,7494
Eco,37.200000,46745


In [14]:
oldest = result_5.loc[result_5['avg_age'].idxmax()]
print(f"{oldest['Class']} passengers are the oldest on average at {oldest['avg_age']} years")

Business passengers are the oldest on average at 41.6 years


## 5. Overall Satisfaction and Loyalty

This section looks at the satisfaction outcome itself: overall rates, how loyalty and travel purpose relate to it, and where dissatisfaction concentrates.

### Q6. What is the overall percentage of satisfied passengers compared to neutral or dissatisfied passengers?

In [15]:
query_6 = '''
SELECT
    satisfaction,
    COUNT(*) AS passenger_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM flights), 2) AS pct_of_total
FROM flights
GROUP BY satisfaction
ORDER BY passenger_count DESC;
'''
result_6 = run_sql(query_6, conn)
show_result(result_6, 'Overall satisfaction split')

satisfaction,passenger_count,pct_of_total
neutral or dissatisfied,58879,56.670000
satisfied,45025,43.330000


In [16]:
satisfied_pct = result_6[result_6['satisfaction'] == 'satisfied']['pct_of_total'].iloc[0]
print(f"{satisfied_pct}% of passengers report being satisfied overall")

43.33% of passengers report being satisfied overall


### Q7. Among passengers who identify as Loyal Customers, what percentage actually report being neutral or dissatisfied with their flight?

In [17]:
query_7 = '''
SELECT
    ROUND(100.0 * SUM(CASE WHEN satisfaction = 'neutral or dissatisfied' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_dissatisfied_among_loyal,
    COUNT(*) AS loyal_customer_count
FROM flights
WHERE "Customer Type" = 'Loyal Customer';
'''
result_7 = run_sql(query_7, conn)
show_result(result_7, 'Dissatisfaction rate among loyal customers')

pct_dissatisfied_among_loyal,loyal_customer_count
52.270000,84923


In [18]:
pct = result_7['pct_dissatisfied_among_loyal'].iloc[0]
print(f"{pct}% of loyal customers still report being neutral or dissatisfied")

52.27% of loyal customers still report being neutral or dissatisfied


### Q8. Does taking a personal trip versus a business trip significantly change the likelihood of a passenger being satisfied?

In [19]:
query_8 = '''
SELECT
    "Type of Travel",
    COUNT(*) AS passenger_count,
    ROUND(100.0 * SUM(CASE WHEN satisfaction = 'satisfied' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_satisfied
FROM flights
GROUP BY "Type of Travel"
ORDER BY pct_satisfied DESC;
'''
result_8 = run_sql(query_8, conn)
show_result(result_8, 'Satisfaction rate by travel purpose')

Type of Travel,passenger_count,pct_satisfied
Business travel,71655,58.260000
Personal Travel,32249,10.170000


In [20]:
gap = result_8.set_index('Type of Travel')['pct_satisfied']
print(f"Business travel satisfaction is {gap['Business travel']}% versus {gap['Personal Travel']}% for personal travel, a gap of {round(gap['Business travel'] - gap['Personal Travel'], 2)} points")

Business travel satisfaction is 58.26% versus 10.17% for personal travel, a gap of 48.09 points


### Q9. Which specific combination of passenger demographics yields the highest rate of dissatisfaction?

In [21]:
query_9 = '''
SELECT
    CASE
        WHEN Age < 18 THEN '0-17'
        WHEN Age < 30 THEN '18-29'
        WHEN Age < 45 THEN '30-44'
        WHEN Age < 60 THEN '45-59'
        ELSE '60+'
    END AS age_group,
    Gender,
    "Customer Type",
    COUNT(*) AS passenger_count,
    ROUND(100.0 * SUM(CASE WHEN satisfaction = 'neutral or dissatisfied' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_dissatisfied
FROM flights
GROUP BY age_group, Gender, "Customer Type"
HAVING COUNT(*) >= 200
ORDER BY pct_dissatisfied DESC
LIMIT 5;
'''
result_9 = run_sql(query_9, conn)
show_result(result_9, 'Top 5 most dissatisfied demographic combinations')

age_group,Gender,Customer Type,passenger_count,pct_dissatisfied
30-44,Female,disloyal Customer,3098,88.150000
30-44,Male,disloyal Customer,2679,86.450000
60+,Female,disloyal Customer,227,86.340000
0-17,Female,Loyal Customer,3454,84.680000
0-17,Male,Loyal Customer,3355,84.320000


In [22]:
top = result_9.iloc[0]
print(f"The most dissatisfied segment is {top['age_group']} {top['Gender']} {top['Customer Type']} passengers at {top['pct_dissatisfied']}% dissatisfaction")

The most dissatisfied segment is 30-44 Female disloyal Customer passengers at 88.15% dissatisfaction


### Q10. Is there a specific threshold of flight distance where overall customer satisfaction begins to noticeably drop?

In [23]:
query_10 = '''
SELECT
    CASE
        WHEN "Flight Distance" < 500 THEN '0-499'
        WHEN "Flight Distance" < 1000 THEN '500-999'
        WHEN "Flight Distance" < 1500 THEN '1000-1499'
        WHEN "Flight Distance" < 2000 THEN '1500-1999'
        WHEN "Flight Distance" < 2500 THEN '2000-2499'
        WHEN "Flight Distance" < 3000 THEN '2500-2999'
        ELSE '3000+'
    END AS distance_bracket,
    COUNT(*) AS passenger_count,
    ROUND(100.0 * SUM(CASE WHEN satisfaction = 'satisfied' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_satisfied
FROM flights
GROUP BY distance_bracket
ORDER BY MIN("Flight Distance");
'''
result_10 = run_sql(query_10, conn)
show_result(result_10, 'Satisfaction rate by flight distance bracket')

distance_bracket,passenger_count,pct_satisfied
0-499,32156,33.310000
500-999,27898,32.540000
1000-1499,12340,36.190000
1500-1999,10084,58.030000
2000-2499,7584,62.680000
2500-2999,5569,67.480000
3000+,8273,77.420000


In [24]:
print(f"Satisfaction rises with distance, from {result_10['pct_satisfied'].iloc[0]}% in the {result_10['distance_bracket'].iloc[0]} mile bracket to {result_10['pct_satisfied'].iloc[-1]}% in the {result_10['distance_bracket'].iloc[-1]} mile bracket, with the sharpest jump around 1500 miles")

Satisfaction rises with distance, from 33.31% in the 0-499 mile bracket to 77.42% in the 3000+ mile bracket, with the sharpest jump around 1500 miles


## 6. Service Ratings and Performance

This section digs into the 14 individual service ratings: which ones matter most, how they differ by class and travel purpose, and how they relate to overall satisfaction.

### Q11. Which specific inflight service has the highest average rating, and which has the lowest across the entire dataset?

In [25]:
query_11 = '\nUNION ALL\n'.join(
    f'SELECT \'{c}\' AS service, ROUND(AVG("{c}"), 3) AS avg_rating FROM flights' for c in service_cols
) + '\nORDER BY avg_rating DESC;'
result_11 = run_sql(query_11, conn)
show_result(result_11, 'Average rating by service, highest to lowest')

service,avg_rating
Inflight service,3.640000
Baggage handling,3.632000
Seat comfort,3.439000
On-board service,3.382000
Inflight entertainment,3.358000
Leg room service,3.351000
Checkin service,3.304000
Cleanliness,3.286000
Online boarding,3.250000
Food and drink,3.202000


In [26]:
best = result_11.iloc[0]
worst = result_11.iloc[-1]
print(f"{best['service']} has the highest average rating at {best['avg_rating']}, while {worst['service']} has the lowest at {worst['avg_rating']}")

Inflight service has the highest average rating at 3.64, while Inflight wifi service has the lowest at 2.73


### Q12. Is there a significant difference in average Seat comfort ratings between Eco class and Business class passengers?

In [27]:
query_12 = '''
SELECT
    Class,
    ROUND(AVG("Seat comfort"), 3) AS avg_seat_comfort,
    COUNT(*) AS passenger_count
FROM flights
WHERE Class IN ('Eco', 'Business')
GROUP BY Class;
'''
result_12 = run_sql(query_12, conn)
show_result(result_12, 'Seat comfort rating, Eco versus Business')

Class,avg_seat_comfort,passenger_count
Business,3.761000,49665
Eco,3.139000,46745


In [28]:
biz = result_12[result_12['Class'] == 'Business']['avg_seat_comfort'].iloc[0]
eco = result_12[result_12['Class'] == 'Eco']['avg_seat_comfort'].iloc[0]
print(f"Business class rates seat comfort at {biz} versus {eco} for Eco, a difference of {round(biz - eco, 3)} points")

Business class rates seat comfort at 3.761 versus 3.139 for Eco, a difference of 0.622 points


### Q13. How do Ease of Online booking ratings compare between passengers taking Personal Travel versus Business Travel?

In [29]:
query_13 = '''
SELECT
    "Type of Travel",
    ROUND(AVG("Ease of Online booking"), 3) AS avg_ease_of_online_booking,
    COUNT(*) AS passenger_count
FROM flights
GROUP BY "Type of Travel"
ORDER BY avg_ease_of_online_booking DESC;
'''
result_13 = run_sql(query_13, conn)
show_result(result_13, 'Ease of online booking rating by travel purpose')

Type of Travel,avg_ease_of_online_booking,passenger_count
Business travel,2.882000,71655
Personal Travel,2.478000,32249


In [30]:
gap = result_13.set_index('Type of Travel')['avg_ease_of_online_booking']
print(f"Business travel rates online booking ease at {gap['Business travel']} versus {gap['Personal Travel']} for personal travel")

Business travel rates online booking ease at 2.882 versus 2.478 for personal travel


### Q14. Which service rating has the strongest statistical correlation with the final overall satisfaction label?

In [31]:
corr_parts = []
for c in service_cols:
    corr_parts.append(f'''
SELECT '{c}' AS service,
    ROUND(
        (COUNT(*) * SUM("{c}" * satisfaction_flag) - SUM("{c}") * SUM(satisfaction_flag)) /
        (SQRT(COUNT(*) * SUM("{c}" * "{c}") - SUM("{c}") * SUM("{c}")) *
         SQRT(COUNT(*) * SUM(satisfaction_flag * satisfaction_flag) - SUM(satisfaction_flag) * SUM(satisfaction_flag))), 4
    ) AS correlation_with_satisfaction
FROM flights_flagged
''')
query_14 = '\nUNION ALL\n'.join(corr_parts) + '\nORDER BY correlation_with_satisfaction DESC;'
result_14 = run_sql(query_14, conn)
show_result(result_14, 'Correlation of each service rating with overall satisfaction')

service,correlation_with_satisfaction
Online boarding,0.503600
Inflight entertainment,0.398100
Seat comfort,0.349500
On-board service,0.322400
Leg room service,0.313100
Cleanliness,0.305200
Inflight wifi service,0.284200
Baggage handling,0.247700
Inflight service,0.244700
Checkin service,0.236200


In [32]:
top = result_14.iloc[0]
print(f"{top['service']} has the strongest correlation with overall satisfaction at r = {top['correlation_with_satisfaction']}")

Online boarding has the strongest correlation with overall satisfaction at r = 0.5036


### Q15. What proportion of customers who rated Inflight entertainment a perfect 5 still ended up overall neutral or dissatisfied?

In [33]:
query_15 = '''
SELECT
    ROUND(100.0 * SUM(CASE WHEN satisfaction = 'neutral or dissatisfied' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_dissatisfied_despite_perfect_entertainment,
    COUNT(*) AS passenger_count
FROM flights
WHERE "Inflight entertainment" = 5;
'''
result_15 = run_sql(query_15, conn)
show_result(result_15, 'Dissatisfaction rate despite a perfect entertainment score')

pct_dissatisfied_despite_perfect_entertainment,passenger_count
35.160000,25213


In [34]:
pct = result_15['pct_dissatisfied_despite_perfect_entertainment'].iloc[0]
print(f"{pct}% of passengers who gave a perfect 5 for inflight entertainment still ended up neutral or dissatisfied overall")

35.16% of passengers who gave a perfect 5 for inflight entertainment still ended up neutral or dissatisfied overall


### Q16. Are passengers who rate Online boarding highly also statistically more likely to rate Checkin service highly?

In [35]:
query_16 = '''
SELECT
    'Online boarding rated 4 or 5' AS segment,
    ROUND(100.0 * SUM(CASE WHEN "Checkin service" >= 4 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_checkin_service_high
FROM flights
WHERE "Online boarding" >= 4
UNION ALL
SELECT
    'Overall population',
    ROUND(100.0 * SUM(CASE WHEN "Checkin service" >= 4 THEN 1 ELSE 0 END) / COUNT(*), 2)
FROM flights;
'''
result_16 = run_sql(query_16, conn)
show_result(result_16, 'Checkin service rating high, by online boarding segment')

segment,pct_checkin_service_high
Online boarding rated 4 or 5,56.400000
Overall population,47.810000


In [36]:
gap = result_16.set_index('segment')['pct_checkin_service_high']
print(f"Passengers who rate online boarding 4 or 5 also rate checkin service 4 or 5 {gap['Online boarding rated 4 or 5']}% of the time, versus {gap['Overall population']}% across the overall population")

Passengers who rate online boarding 4 or 5 also rate checkin service 4 or 5 56.4% of the time, versus 47.81% across the overall population


### Q17. Do loyal customers give harsher ratings for Inflight service and Baggage handling compared to disloyal customers?

In [37]:
query_17 = '''
SELECT
    "Customer Type",
    ROUND(AVG("Inflight service"), 3) AS avg_inflight_service,
    ROUND(AVG("Baggage handling"), 3) AS avg_baggage_handling
FROM flights
GROUP BY "Customer Type";
'''
result_17 = run_sql(query_17, conn)
show_result(result_17, 'Inflight service and baggage handling rating by customer type')

Customer Type,avg_inflight_service,avg_baggage_handling
Loyal Customer,3.628000,3.618000
disloyal Customer,3.697000,3.694000


In [38]:
loyal = result_17[result_17['Customer Type'] == 'Loyal Customer'].iloc[0]
disloyal = result_17[result_17['Customer Type'] == 'disloyal Customer'].iloc[0]
print(f"Loyal customers rate inflight service {loyal['avg_inflight_service']} and baggage handling {loyal['avg_baggage_handling']}, both slightly below disloyal customers at {disloyal['avg_inflight_service']} and {disloyal['avg_baggage_handling']}")

Loyal customers rate inflight service 3.628 and baggage handling 3.618, both slightly below disloyal customers at 3.697 and 3.694


## 7. Delays and Operational Impact

This section examines departure and arrival delays: how they vary by class and distance, how they affect satisfaction and specific ratings, and how passengers behave under extreme delay.

### Q18. What is the average departure delay and arrival delay, and do these averages differ significantly by travel class?

In [39]:
query_18 = '''
SELECT
    Class,
    ROUND(AVG("Departure Delay in Minutes"), 2) AS avg_departure_delay,
    ROUND(AVG("Arrival Delay in Minutes"), 2) AS avg_arrival_delay
FROM flights
GROUP BY Class
ORDER BY avg_arrival_delay DESC;
'''
result_18 = run_sql(query_18, conn)
show_result(result_18, 'Average delay by travel class')

Class,avg_departure_delay,avg_arrival_delay
Eco Plus,15.430000,16.090000
Eco,15.160000,15.670000
Business,14.400000,14.580000


In [40]:
top = result_18.iloc[0]
print(f"{top['Class']} passengers see the longest average delays at {top['avg_departure_delay']} minutes departure and {top['avg_arrival_delay']} minutes arrival")

Eco Plus passengers see the longest average delays at 15.43 minutes departure and 16.09 minutes arrival


### Q19. How does the total delay vary based on flight distance categories?

In [41]:
query_19 = '''
SELECT
    CASE
        WHEN "Flight Distance" < 1000 THEN 'Short-haul (under 1000)'
        WHEN "Flight Distance" < 2500 THEN 'Medium-haul (1000-2499)'
        ELSE 'Long-haul (2500+)'
    END AS distance_category,
    ROUND(AVG("Departure Delay in Minutes" + COALESCE("Arrival Delay in Minutes", 0)), 2) AS avg_total_delay,
    COUNT(*) AS passenger_count
FROM flights
GROUP BY distance_category
ORDER BY MIN("Flight Distance");
'''
result_19 = run_sql(query_19, conn)
show_result(result_19, 'Average total delay by distance category')

distance_category,avg_total_delay,passenger_count
Short-haul (under 1000),30.030000,60054
Medium-haul (1000-2499),30.040000,30008
Long-haul (2500+),29.400000,13842


In [42]:
spread = round(result_19['avg_total_delay'].max() - result_19['avg_total_delay'].min(), 2)
print(f"Average total delay is fairly stable across distance categories, with only a {spread} minute spread between the highest and lowest")

Average total delay is fairly stable across distance categories, with only a 0.64 minute spread between the highest and lowest


### Q20. At what arrival delay bracket does the probability of a passenger being satisfied drop below 50%?

In [43]:
query_20 = '''
SELECT
    CASE
        WHEN "Arrival Delay in Minutes" IS NULL THEN 'Unknown'
        WHEN "Arrival Delay in Minutes" = 0 THEN '0 (on time)'
        WHEN "Arrival Delay in Minutes" <= 15 THEN '1-15 min'
        WHEN "Arrival Delay in Minutes" <= 30 THEN '16-30 min'
        WHEN "Arrival Delay in Minutes" <= 60 THEN '31-60 min'
        ELSE '60+ min'
    END AS arrival_delay_bracket,
    COUNT(*) AS passenger_count,
    ROUND(100.0 * SUM(CASE WHEN satisfaction = 'satisfied' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_satisfied
FROM flights
GROUP BY arrival_delay_bracket
ORDER BY MIN(COALESCE("Arrival Delay in Minutes", -1));
'''
result_20 = run_sql(query_20, conn)
show_result(result_20, 'Satisfaction rate by arrival delay bracket')

arrival_delay_bracket,passenger_count,pct_satisfied
Unknown,310,41.290000
0 (on time),58159,47.280000
1-15 min,21800,41.300000
16-30 min,8973,35.270000
31-60 min,7335,35.640000
60+ min,7327,35.720000


In [44]:
below_50 = result_20[result_20['pct_satisfied'] < 50]
on_time_rate = result_20[result_20['arrival_delay_bracket'] == '0 (on time)']['pct_satisfied'].iloc[0]
print(f"Satisfaction never reaches 50% in any bracket of this dataset, even on-time flights sit at only {on_time_rate}%, so delay alone does not explain the satisfaction gap")

Satisfaction never reaches 50% in any bracket of this dataset, even on-time flights sit at only 47.28%, so delay alone does not explain the satisfaction gap


### Q21. Are Departure/Arrival time convenient ratings demonstrably lower for flights with an arrival delay of more than 45 minutes?

In [45]:
query_21 = '''
SELECT
    CASE WHEN "Arrival Delay in Minutes" > 45 THEN 'Delayed 45+ min' ELSE 'On time or delayed up to 45 min' END AS delay_group,
    ROUND(AVG("Departure/Arrival time convenient"), 3) AS avg_time_convenient_rating,
    COUNT(*) AS passenger_count
FROM flights
WHERE "Arrival Delay in Minutes" IS NOT NULL
GROUP BY delay_group;
'''
result_21 = run_sql(query_21, conn)
show_result(result_21, 'Time convenience rating by delay group')

delay_group,avg_time_convenient_rating,passenger_count
Delayed 45+ min,3.043000,10066
On time or delayed up to 45 min,3.062000,93528


In [46]:
gap = result_21.set_index('delay_group')['avg_time_convenient_rating']
print(f"Departure and arrival time convenience ratings are nearly flat regardless of delay, {gap['Delayed 45+ min']} when delayed 45 or more minutes versus {gap['On time or delayed up to 45 min']} otherwise")

Departure and arrival time convenience ratings are nearly flat regardless of delay, 3.043 when delayed 45 or more minutes versus 3.062 otherwise


### Q22. What percentage of flights have an arrival delay greater than zero but a departure delay of zero?

In [47]:
query_22 = '''
SELECT
    ROUND(100.0 * SUM(CASE WHEN "Arrival Delay in Minutes" > 0 AND "Departure Delay in Minutes" = 0 THEN 1 ELSE 0 END) / COUNT(*), 3) AS pct_air_only_delay,
    SUM(CASE WHEN "Arrival Delay in Minutes" > 0 AND "Departure Delay in Minutes" = 0 THEN 1 ELSE 0 END) AS flight_count
FROM flights
WHERE "Arrival Delay in Minutes" IS NOT NULL;
'''
result_22 = run_sql(query_22, conn)
show_result(result_22, 'Flights that depart on time but arrive late')

pct_air_only_delay,flight_count
10.708000,11093


In [48]:
pct = result_22['pct_air_only_delay'].iloc[0]
print(f"{pct}% of flights depart on time but still land late, meaning the delay accumulated entirely in the air")

10.708% of flights depart on time but still land late, meaning the delay accumulated entirely in the air


### Q23. How does the gap between departure delay and arrival delay correlate with passenger ratings for On-board service?

In [49]:
query_23 = '''
WITH derived AS (
    SELECT
        ("Departure Delay in Minutes" - "Arrival Delay in Minutes") AS time_made_up_in_air,
        "On-board service" AS onboard_rating
    FROM flights
    WHERE "Arrival Delay in Minutes" IS NOT NULL
)
SELECT
    ROUND(
        (COUNT(*) * SUM(time_made_up_in_air * onboard_rating) - SUM(time_made_up_in_air) * SUM(onboard_rating)) /
        (SQRT(COUNT(*) * SUM(time_made_up_in_air * time_made_up_in_air) - SUM(time_made_up_in_air) * SUM(time_made_up_in_air)) *
         SQRT(COUNT(*) * SUM(onboard_rating * onboard_rating) - SUM(onboard_rating) * SUM(onboard_rating))), 4
    ) AS correlation_time_made_up_vs_onboard_service
FROM derived;
'''
result_23 = run_sql(query_23, conn)
show_result(result_23, 'Correlation between time made up in the air and on-board service rating')

correlation_time_made_up_vs_onboard_service
0.016200


In [50]:
r = result_23['correlation_time_made_up_vs_onboard_service'].iloc[0]
print(f"The correlation between time made up in the air and on-board service rating is only r = {r}, essentially no relationship")

The correlation between time made up in the air and on-board service rating is only r = 0.0162, essentially no relationship


### Q24. Among passengers who experienced extreme delays, which inflight service received the highest rating?

In [51]:
top5_union = '\nUNION ALL\n'.join(
    f'SELECT \'{c}\' AS service, ROUND(AVG("{c}"), 3) AS avg_rating FROM top5' for c in service_cols
)
query_24 = f'''
WITH ranked AS (
    SELECT *, PERCENT_RANK() OVER (ORDER BY "Arrival Delay in Minutes") AS pct_rank
    FROM flights
    WHERE "Arrival Delay in Minutes" IS NOT NULL
),
top5 AS (
    SELECT * FROM ranked WHERE pct_rank >= 0.95
)
{top5_union}
ORDER BY avg_rating DESC;
'''
result_24 = run_sql(query_24, conn)
show_result(result_24, 'Average service rating among the top 5% most delayed passengers')

service,avg_rating
Baggage handling,3.614000
Leg room service,3.433000
Inflight service,3.399000
Seat comfort,3.306000
Inflight entertainment,3.247000
Cleanliness,3.240000
On-board service,3.237000
Checkin service,3.226000
Online boarding,3.173000
Food and drink,3.064000


In [52]:
top = result_24.iloc[0]
print(f"Among the most delayed 5% of passengers, {top['service']} still earns the highest average rating at {top['avg_rating']}")

Among the most delayed 5% of passengers, Baggage handling still earns the highest average rating at 3.614


### Q25. What is the median flight distance for passengers who rated Leg room service a 1 or 2, versus those who rated it a 4 or 5?

In [53]:
query_25 = '''
WITH low_rated AS (
    SELECT "Flight Distance" AS distance,
        ROW_NUMBER() OVER (ORDER BY "Flight Distance") AS rn,
        COUNT(*) OVER () AS total
    FROM flights
    WHERE "Leg room service" IN (1, 2)
),
high_rated AS (
    SELECT "Flight Distance" AS distance,
        ROW_NUMBER() OVER (ORDER BY "Flight Distance") AS rn,
        COUNT(*) OVER () AS total
    FROM flights
    WHERE "Leg room service" IN (4, 5)
)
SELECT 'Leg room rated 1 or 2' AS segment, ROUND(AVG(distance), 1) AS median_flight_distance
FROM low_rated
WHERE rn IN ((total + 1) / 2, (total + 2) / 2)
UNION ALL
SELECT 'Leg room rated 4 or 5', ROUND(AVG(distance), 1)
FROM high_rated
WHERE rn IN ((total + 1) / 2, (total + 2) / 2);
'''
result_25 = run_sql(query_25, conn)
show_result(result_25, 'Median flight distance by leg room rating')

segment,median_flight_distance
Leg room rated 1 or 2,721.000000
Leg room rated 4 or 5,946.000000


In [54]:
gap = result_25.set_index('segment')['median_flight_distance']
print(f"Passengers who rate leg room 1 or 2 fly a median of {gap['Leg room rated 1 or 2']} miles, versus {gap['Leg room rated 4 or 5']} miles for those who rate it 4 or 5")

Passengers who rate leg room 1 or 2 fly a median of 721.0 miles, versus 946.0 miles for those who rate it 4 or 5


## 8. Key Takeaways

- Only 43.33% of passengers report being satisfied overall, while 56.67% are neutral or dissatisfied.
- Business travel satisfaction (58.26%) is dramatically higher than personal travel satisfaction (10.17%), a gap of over 48 points.
- Online boarding has the strongest correlation with overall satisfaction (r = 0.5036) among all 14 service ratings, well ahead of inflight entertainment (r = 0.3981).
- Business class passengers are the oldest on average (41.6 years) and Loyal Customers fly nearly twice as far on average as disloyal customers (1,295.6 vs 714.5 miles).
- Delay length barely moves satisfaction after the first 15 minutes, and even on-time flights sit at only 47.28% satisfied, so delay alone does not explain the overall satisfaction gap.
- 35.16% of passengers who rated inflight entertainment a perfect 5 still ended up dissatisfied overall, showing no single service fully drives the outcome.

In [55]:
conn.close()